# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant metadata dataset using the `mlcroissant` library. We will step through loading metadata, exploring record sets and fields (by unique `@id`), extracting data, conducting EDA, and visualizing the results.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and visualization dependencies are installed
!pip install mlcroissant
!pip install matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, their `@id`, fields, and selection for further analysis.

In [ ]:
# List available Record Sets and their Fields by @id

record_set_ids = []
try:
    for rs in meta.record_sets:
        print(f"RecordSet @id: {rs.id}")
        record_set_ids.append(rs.id)
        print(f"  Name: {rs.name}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id}  |  name: {field.name}  |  dataType: {field.data_type}")
        print()
except Exception as e:
    print("Error accessing record sets. Make sure the Croissant schema exposes record sets.")

print(f"All record set @ids: {record_set_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# If there are multiple record sets, select one to proceed with.
import warnings
warnings.filterwarnings("ignore")
dataframes = {}

# Use the first record set for analysis (override if needed for specific @id)
if len(record_set_ids) > 0:
    selected_recordset_id = record_set_ids[0]
    print(f"Using record set @id: {selected_recordset_id}")
    records = list(dataset.records(record_set=selected_recordset_id))
    df = pd.DataFrame(records)
    dataframes[selected_recordset_id] = df
    print("Sample columns:", df.columns.tolist())
    display(df.head())
else:
    raise RuntimeError("No record sets available in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize a numeric field, and group/categorize if appropriate.
<br>
We'll pick one of the numeric fields identified above (such as 'coverage_percent', 'peptide_count', or 'MW'), referencing the actual column name as it appears in the DataFrame (typically derived from the field `@id` or `name`).

In [ ]:
# Explore column names and select a numeric field. Adjust as needed based on DataFrame inspection.

# For demonstration, we guess commonly encountered fields: MW (molecular weight), peptide_count, etc.
# Replace with the column name as shown in the DataFrame above if different.
numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or sum(c.isdigit() for c in str(df[col].iloc[0])[:6]) > 2]
print('Numeric candidates:', numeric_fields)

# Try MW (molecular weight) or fallback to another
candidate_fields = ['MW', 'molecular_weight', 'peptide_count', 'coverage_percent']
numeric_field = None
for field in candidate_fields:
    if field in df.columns:
        numeric_field = field
        break
if numeric_field is None and len(numeric_fields) > 0:
    numeric_field = numeric_fields[0]
if numeric_field is None:
    raise RuntimeError("No suitable numeric field found for demonstration.")
print(f"Selected numeric field: {numeric_field}")

# Filter to records with numeric_field > threshold
threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered {len(filtered_df)}/{len(df)} records with {numeric_field} > {threshold:.2f}.")
print(filtered_df[[numeric_field]].head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a 'description' or 'accession' field for demonstration
group_fields = [f for f in ['description', 'accession'] if f in df.columns]
if group_fields:
    group_field = group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the effect of normalization and filtering.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot original vs normalized numeric field
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df[numeric_field], kde=True, bins=30, color='skyblue')
plt.title(f'Distribution of {numeric_field} (All)')

plt.subplot(1, 2, 2)
sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, bins=30, color='orange')
plt.title(f'Normalized {numeric_field} (Filtered > Q3)')

plt.tight_layout()
plt.show()

# If group_field exists, visualize mean by group
if 'group_field' in locals():
    plt.figure(figsize=(10, 6))
    topn = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    sns.barplot(y=group_field, x=numeric_field, data=topn, palette="viridis")
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel(group_field)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated stepwise exploration of the FAIR^2 Croissant-formatted dataset using `mlcroissant` by referencing all entities with their `@id`, loading into dataframes, filtering and normalizing a numeric field, and visualizing field distributions. This foundation enables further repeated, reliable analysis with high traceability to dataset schema definitions.

Key observations:
- The dataset contains comprehensive protein records with well-defined fields and IDs.
- The distribution of numeric properties such as molecular weight is right-skewed, with a subset of proteins above the upper quantile.
- All data processing steps reference Croissant `@id` entities, promoting precise, reproducible science.